In [3]:
# ==============================================================
# Benchmark (CLEAN + RF + LGBM):
# Prophet, SARIMA/SARIMAX, XGBoost-direct, RandomForest-direct, LightGBM-direct,
# Hybrid-Prophet, Hybrid-SARIMA
# Data: veri_matrisi_final_sales_orders_stock_calendar_lags_fx.csv
# Splits: TRAIN_FULL=...2024-12 | VAL=2024-07..2024-12 | TEST=2025-01..2025-07
# ==============================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import product

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
try:
    from lightgbm import LGBMRegressor
    LGBM_AVAILABLE = True
except Exception:
    LGBM_AVAILABLE = False
    print("[UYARI] lightgbm import edilemedi. LightGBM sonuçları üretilmeyecek.")

from prophet import Prophet
from statsmodels.tsa.statespace.sarimax import SARIMAX

# ------------------ Config ------------------
CSV_PATH   = "veri_matrisi_final_sales_orders_stock_calendar_lags_fx.csv"

TRAIN_END  = pd.Timestamp("2024-12-01")
VAL_START  = pd.Timestamp("2024-07-01")
VAL_END    = pd.Timestamp("2024-12-01")
TEST_START = pd.Timestamp("2025-01-01")
TEST_END   = pd.Timestamp("2025-07-01")

RANDOM_STATE = 42

# ------------------ Utils ------------------
def mae_rmse_mape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    denom = np.where(y_true == 0, 1, y_true)
    mape = np.mean(np.abs((y_true - y_pred) / denom)) * 100
    return mae, rmse, mape

def select_x_cols(df_cols):
    preferred = [
        "month","year","quarter",
        "is_Fall","is_Spring","is_Summer","is_Winter",
        "orders","stock","eurtry","usdtry","orders_ratio",
        "y_lag1","y_lag12","orders_lag1","orders_lag12","stock_lag1","stock_lag12",
    ]
    return [c for c in preferred if c in df_cols]

def clean_xy(X: pd.DataFrame, y: pd.Series):
    """Etikette NaN/Inf satırları düş, özellikleri normalize et."""
    y_np = y.to_numpy()
    mask_label_ok = np.isfinite(y_np)
    X = X.loc[mask_label_ok]
    y = y.loc[mask_label_ok]
    X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
    return X, y

# ------------------ Load data ------------------
df = pd.read_csv(CSV_PATH, parse_dates=["ds"]).sort_values("ds").reset_index(drop=True)

# numeric safety
num_cols = ["y","orders","stock","eurtry","usdtry","orders_ratio",
            "y_lag1","y_lag12","orders_lag1","orders_lag12","stock_lag1","stock_lag12"]
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# splits
mask_train_full = (df["ds"] <= TRAIN_END)
mask_val        = (df["ds"] >= VAL_START) & (df["ds"] <= VAL_END)
mask_train_in   = (df["ds"] <  VAL_START)
mask_test       = (df["ds"] >= TEST_START) & (df["ds"] <= TEST_END)

train_in_df   = df.loc[mask_train_in].copy()
val_df        = df.loc[mask_val].copy()
train_full_df = df.loc[mask_train_full].copy()
test_df       = df.loc[mask_test].copy()

y_test_true   = test_df["y"].to_numpy()
dates_test    = test_df["ds"].to_numpy()

# ------------------ Prophet (opt) ------------------
def optimize_prophet(df_train_in, df_val):
    grid = {
        "changepoint_prior_scale": [0.01, 0.05, 0.1],
        "seasonality_mode": ["additive", "multiplicative"],
        "seasonality_prior_scale": [1.0, 5.0]
    }
    best, best_mae = None, np.inf
    for cps, smode, sps in product(grid["changepoint_prior_scale"], grid["seasonality_mode"], grid["seasonality_prior_scale"]):
        try:
            m = Prophet(yearly_seasonality=True, weekly_seasonality=False,
                        changepoint_prior_scale=cps, seasonality_mode=smode,
                        seasonality_prior_scale=sps)
            m.fit(df_train_in[["ds","y"]])
            future = m.make_future_dataframe(periods=len(df_val), freq="MS")
            fcst = m.predict(future)[["ds","yhat"]]
            y_pred = fcst[fcst["ds"].isin(df_val["ds"])]["yhat"].to_numpy()
            y_true = df_val["y"].to_numpy()
            mae, _, _ = mae_rmse_mape(y_true, y_pred)
            if mae < best_mae:
                best_mae = mae
                best = {"changepoint_prior_scale":cps, "seasonality_mode":smode, "seasonality_prior_scale":sps}
        except Exception:
            continue
    return best

prophet_best = optimize_prophet(train_in_df, val_df)
m_prophet = Prophet(yearly_seasonality=True, weekly_seasonality=False, **prophet_best)
m_prophet.fit(train_full_df[["ds","y"]])
future_full = m_prophet.make_future_dataframe(periods=len(test_df), freq="MS")
fcst_prophet = m_prophet.predict(future_full)[["ds","yhat"]]
prophet_test = fcst_prophet[fcst_prophet["ds"].isin(test_df["ds"])]["yhat"].to_numpy()

# ------------------ SARIMA/SARIMAX (opt) ------------------
exog_cols = [c for c in ["orders","stock","eurtry","usdtry"] if c in df.columns]

series_train_in   = train_in_df.set_index("ds")["y"]
series_val        = val_df.set_index("ds")["y"]
series_train_full = train_full_df.set_index("ds")["y"]

exog_train_in   = train_in_df.set_index("ds")[exog_cols] if exog_cols else None
exog_val        = val_df.set_index("ds")[exog_cols] if exog_cols else None
exog_train_full = train_full_df.set_index("ds")[exog_cols] if exog_cols else None
exog_test       = test_df.set_index("ds")[exog_cols] if exog_cols else None

def optimize_sarima(y_train_in, y_val, exog_train_in=None, exog_val=None):
    pdq_list  = [(p,1,q) for p in [0,1,2] for q in [0,1,2]]
    PDQ_list  = [(P,1,Q,12) for P in [0,1] for Q in [0,1]]
    best, best_mae = None, np.inf
    for order in pdq_list:
        for sorder in PDQ_list:
            try:
                model = SARIMAX(y_train_in, order=order, seasonal_order=sorder,
                                exog=exog_train_in, enforce_stationarity=False, enforce_invertibility=False)
                res = model.fit(disp=False)
                fc = res.get_forecast(steps=len(y_val), exog=exog_val).predicted_mean.to_numpy()
                mae, _, _ = mae_rmse_mape(y_val.to_numpy(), fc)
                if mae < best_mae:
                    best_mae = mae
                    best = {"order":order, "seasonal_order":sorder}
            except Exception:
                continue
    return best

sarima_best = optimize_sarima(series_train_in, series_val, exog_train_in, exog_val)

sarima_model = SARIMAX(series_train_full, order=sarima_best["order"], seasonal_order=sarima_best["seasonal_order"],
                       exog=exog_train_full, enforce_stationarity=False, enforce_invertibility=False)
sarima_res = sarima_model.fit(disp=False)
sarima_test = sarima_res.get_forecast(steps=len(test_df), exog=exog_test).predicted_mean.to_numpy()

# ------------------ XGBoost-direct (opt) ------------------
X_cols = select_x_cols(df.columns)

X_train_in   = train_in_df[X_cols].copy()
y_train_in   = train_in_df["y"].copy()
X_val        = val_df[X_cols].copy()
y_val        = val_df["y"].copy()
X_train_in, y_train_in = clean_xy(X_train_in, y_train_in)
X_val,      y_val      = clean_xy(X_val,      y_val)

X_train_full = train_full_df[X_cols].copy()
y_train_full = train_full_df["y"].copy()
X_train_full, y_train_full = clean_xy(X_train_full, y_train_full)

X_test = test_df[X_cols].copy().replace([np.inf, -np.inf], np.nan).fillna(0)

def optimize_xgb(X_tr_in, y_tr_in, X_v, y_v):
    grid = {
        "n_estimators": [300, 500],
        "learning_rate": [0.05, 0.1],
        "max_depth": [2, 3, 4],
        "subsample": [0.9],
        "colsample_bytree": [0.9],
        "reg_lambda": [1.0, 2.0]
    }
    best_params, best_mae = None, np.inf
    for n, lr, md, ss, cs, rl in product(grid["n_estimators"], grid["learning_rate"], grid["max_depth"],
                                         grid["subsample"], grid["colsample_bytree"], grid["reg_lambda"]):
        params = dict(n_estimators=n, learning_rate=lr, max_depth=md,
                      subsample=ss, colsample_bytree=cs, reg_lambda=rl,
                      random_state=RANDOM_STATE)
        model = XGBRegressor(**params)
        model.fit(X_tr_in, y_tr_in)
        val_pred = model.predict(X_v)
        mae, _, _ = mae_rmse_mape(y_v, val_pred)
        if mae < best_mae:
            best_mae, best_params = mae, params
    return best_params

xgb_best = optimize_xgb(X_train_in, y_train_in, X_val, y_val)
xgb_direct = XGBRegressor(**xgb_best)
xgb_direct.fit(X_train_full, y_train_full)
xgb_test = xgb_direct.predict(X_test)

# ------------------ RandomForest-direct (opt) ------------------
def optimize_rf(X_tr_in, y_tr_in, X_v, y_v):
    grid = {
        "n_estimators": [200, 500],
        "max_depth": [None, 5, 10],
        "min_samples_split": [2, 5]
    }
    best_params, best_mae = None, np.inf
    for n, md, mss in product(grid["n_estimators"], grid["max_depth"], grid["min_samples_split"]):
        params = dict(n_estimators=n, max_depth=md, min_samples_split=mss,
                      random_state=RANDOM_STATE, n_jobs=-1)
        model = RandomForestRegressor(**params)
        model.fit(X_tr_in, y_tr_in)
        val_pred = model.predict(X_v)
        mae, _, _ = mae_rmse_mape(y_v, val_pred)
        if mae < best_mae:
            best_mae, best_params = mae, params
    return best_params

rf_best = optimize_rf(X_train_in, y_train_in, X_val, y_val)
rf_model = RandomForestRegressor(**rf_best)
rf_model.fit(X_train_full, y_train_full)
rf_test = rf_model.predict(X_test)

# ------------------ LightGBM-direct (opt) ------------------
def optimize_lgbm(X_tr_in, y_tr_in, X_v, y_v):
    grid = {
        "n_estimators": [300, 500],
        "learning_rate": [0.05, 0.1],
        "max_depth": [-1, 5, 10],
        "num_leaves": [31, 63]
    }
    best_params, best_mae = None, np.inf
    for n, lr, md, nl in product(grid["n_estimators"], grid["learning_rate"], grid["max_depth"], grid["num_leaves"]):
        params = dict(n_estimators=n, learning_rate=lr, max_depth=md,
                      num_leaves=nl, random_state=RANDOM_STATE)
        model = LGBMRegressor(**params)
        model.fit(X_tr_in, y_tr_in)
        val_pred = model.predict(X_v)
        mae, _, _ = mae_rmse_mape(y_v, val_pred)
        if mae < best_mae:
            best_mae, best_params = mae, params
    return best_params

if LGBM_AVAILABLE:
    lgbm_best = optimize_lgbm(X_train_in, y_train_in, X_val, y_val)
    lgbm_model = LGBMRegressor(**lgbm_best)
    lgbm_model.fit(X_train_full, y_train_full)
    lgbm_test = lgbm_model.predict(X_test)
else:
    lgbm_best, lgbm_test = None, None

# ------------------ Hybrid-Prophet (Prophet + XGB residual) ------------------
m_prophet_tmp = Prophet(yearly_seasonality=True, weekly_seasonality=False, **prophet_best)
m_prophet_tmp.fit(train_in_df[["ds","y"]])
tmp_future = m_prophet_tmp.make_future_dataframe(periods=len(val_df), freq="MS")
tmp_fcst   = m_prophet_tmp.predict(tmp_future)[["ds","yhat"]]

tmp_merged = df.merge(tmp_fcst.rename(columns={"yhat":"yhat_tmp"}), on="ds", how="left")
tr_in = tmp_merged.loc[mask_train_in].copy()
vl    = tmp_merged.loc[mask_val].copy()
tr_in["residual_tmp"] = tr_in["y"] - tr_in["yhat_tmp"]
vl["residual_tmp"]    = vl["y"] - vl["yhat_tmp"]

# drop NaN residual labels and clean XY
X_cols_res = select_x_cols(df.columns)  # y_lag'ler zaten dahil
tr_in_nonan = tr_in.dropna(subset=["residual_tmp"]).copy()
vl_nonan    = vl.dropna(subset=["residual_tmp"]).copy()

X_tr_in_res = tr_in_nonan[X_cols_res].copy()
y_tr_in_res = tr_in_nonan["residual_tmp"].copy()
X_val_res   = vl_nonan[X_cols_res].copy()
y_val_res   = vl_nonan["residual_tmp"].copy()

X_tr_in_res, y_tr_in_res = clean_xy(X_tr_in_res, y_tr_in_res)
X_val_res,   y_val_res   = clean_xy(X_val_res,   y_val_res)

xgb_res_best_prophet = optimize_xgb(X_tr_in_res, y_tr_in_res, X_val_res, y_val_res)

# Final residual training on TRAIN_FULL (Prophet fitted above => fcst_prophet)
df_with_prophet = df.merge(fcst_prophet.rename(columns={"yhat":"yhat_prophet"}), on="ds", how="left")
train_full_prop = df_with_prophet.loc[mask_train_full].copy()
train_full_prop["residual"] = train_full_prop["y"] - train_full_prop["yhat_prophet"]
tmp_full_nonan = train_full_prop.dropna(subset=["residual"]).copy()

X_tr_full_res = tmp_full_nonan[X_cols_res].copy()
y_tr_full_res = tmp_full_nonan["residual"].copy()
X_tr_full_res, y_tr_full_res = clean_xy(X_tr_full_res, y_tr_full_res)

xgb_res_prophet = XGBRegressor(**xgb_res_best_prophet)
xgb_res_prophet.fit(X_tr_full_res, y_tr_full_res)

X_test_res = df_with_prophet.loc[mask_test, X_cols_res].copy().replace([np.inf, -np.inf], np.nan).fillna(0)
resid_pred_prophet = xgb_res_prophet.predict(X_test_res)
hybrid_prophet_test = prophet_test + resid_pred_prophet

# ------------------ Hybrid-SARIMA (SARIMA + XGB residual) ------------------
# TRAIN_IN kısa fit + VAL residual optimize
sarima_in = SARIMAX(series_train_in, order=sarima_best["order"], seasonal_order=sarima_best["seasonal_order"],
                    exog=exog_train_in, enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
fitted_in = sarima_in.fittedvalues
fc_val    = sarima_in.get_forecast(steps=len(val_df), exog=exog_val).predicted_mean

tmp2 = df.copy()
df_in_fit = pd.DataFrame({"ds": series_train_in.index, "yhat_sarima_inval": fitted_in.values})
tmp2 = tmp2.merge(df_in_fit, on="ds", how="left")
tmp2.loc[tmp2["ds"].isin(val_df["ds"]), "yhat_sarima_inval"] = fc_val.to_numpy()

tr_in2 = tmp2.loc[mask_train_in].copy()
vl2    = tmp2.loc[mask_val].copy()
tr_in2["residual_tmp"] = tr_in2["y"] - tr_in2["yhat_sarima_inval"]
vl2["residual_tmp"]    = vl2["y"] - vl2["yhat_sarima_inval"]

tr_in2_nonan = tr_in2.dropna(subset=["residual_tmp"]).copy()
vl2_nonan    = vl2.dropna(subset=["residual_tmp"]).copy()

X_tr_in_res2 = tr_in2_nonan[X_cols_res].copy()
y_tr_in_res2 = tr_in2_nonan["residual_tmp"].copy()
X_val_res2   = vl2_nonan[X_cols_res].copy()
y_val_res2   = vl2_nonan["residual_tmp"].copy()

X_tr_in_res2, y_tr_in_res2 = clean_xy(X_tr_in_res2, y_tr_in_res2)
X_val_res2,   y_val_res2   = clean_xy(X_val_res2,   y_val_res2)

xgb_res_best_sarima = optimize_xgb(X_tr_in_res2, y_tr_in_res2, X_val_res2, y_val_res2)

# TRAIN_FULL residual fit (SARIMA fitted above on TRAIN_FULL -> sarima_res)
fitted_full_sarima = sarima_res.fittedvalues.reindex(train_full_df.set_index("ds").index)
tmp_full2 = train_full_df.copy()
tmp_full2["yhat_sarima"] = fitted_full_sarima.values
tmp_full2["residual"] = tmp_full2["y"] - tmp_full2["yhat_sarima"]
tmp_full2_nonan = tmp_full2.dropna(subset=["residual"]).copy()

X_tr_full_res2 = tmp_full2_nonan[X_cols_res].copy()
y_tr_full_res2 = tmp_full2_nonan["residual"].copy()
X_tr_full_res2, y_tr_full_res2 = clean_xy(X_tr_full_res2, y_tr_full_res2)

xgb_res_sarima = XGBRegressor(**xgb_res_best_sarima)
xgb_res_sarima.fit(X_tr_full_res2, y_tr_full_res2)

X_test_res2 = test_df[X_cols_res].copy().replace([np.inf, -np.inf], np.nan).fillna(0)
resid_pred_sarima = xgb_res_sarima.predict(X_test_res2)
hybrid_sarima_test = sarima_test + resid_pred_sarima

# ------------------ Metrics & Report ------------------
rows = []
def add_row(name, y_hat):
    mae, rmse, mape = mae_rmse_mape(y_test_true, y_hat)
    rows.append({"Model":name, "MAE":mae, "RMSE":rmse, "MAPE":mape})

add_row("Prophet", prophet_test)
add_row("SARIMA", sarima_test)
add_row("XGB-direct", xgb_test)
add_row("RandomForest", rf_test)
if LGBM_AVAILABLE:
    add_row("LightGBM", lgbm_test)
add_row("Hybrid-Prophet", hybrid_prophet_test)
add_row("Hybrid-SARIMA", hybrid_sarima_test)

res_df = pd.DataFrame(rows).sort_values("MAPE").reset_index(drop=True)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("\n=== Hiperparametreler ===")
print("Prophet:", prophet_best)
print("SARIMA :", sarima_best)
print("XGB-direct:", xgb_best)
print("RF-direct :", rf_best)
if LGBM_AVAILABLE:
    print("LGBM-direct:", lgbm_best)
print("XGB-residual (Prophet):", xgb_res_best_prophet)
print("XGB-residual (SARIMA) :", xgb_res_best_sarima)

print("\n=== TEST (2025-01..07) Sonuçları ===")
print(res_df)

# ------------------ Plot: Actual vs Predictions (TEST) ------------------
plt.figure(figsize=(12,6))
plt.plot(dates_test, y_test_true, "o-", label="Gerçek (Test)")

plt.plot(dates_test, prophet_test, "--", label="Prophet")
plt.plot(dates_test, sarima_test,  "--", label="SARIMA")
plt.plot(dates_test, xgb_test,     "--", label="XGB-direct")
plt.plot(dates_test, rf_test,      "--", label="RandomForest")
if LGBM_AVAILABLE:
    plt.plot(dates_test, lgbm_test,   "--", label="LightGBM")
plt.plot(dates_test, hybrid_prophet_test, "--", label="Hybrid-Prophet")
plt.plot(dates_test, hybrid_sarima_test,  "--", label="Hybrid-SARIMA")

plt.title("Gerçek vs Tahmin • TEST (2025-01..07)")
plt.xlabel("Tarih"); plt.ylabel("Satış"); plt.legend(); plt.tight_layout()
plt.show()


16:45:06 - cmdstanpy - INFO - Chain [1] start processing
16:45:06 - cmdstanpy - INFO - Chain [1] done processing
16:45:06 - cmdstanpy - INFO - Chain [1] start processing
16:45:06 - cmdstanpy - INFO - Chain [1] done processing
16:45:06 - cmdstanpy - INFO - Chain [1] start processing


[UYARI] lightgbm import edilemedi. LightGBM sonuçları üretilmeyecek.


16:45:06 - cmdstanpy - INFO - Chain [1] done processing
16:45:06 - cmdstanpy - INFO - Chain [1] start processing
16:45:06 - cmdstanpy - INFO - Chain [1] done processing
16:45:06 - cmdstanpy - INFO - Chain [1] start processing
16:45:06 - cmdstanpy - INFO - Chain [1] done processing
16:45:06 - cmdstanpy - INFO - Chain [1] start processing
16:45:06 - cmdstanpy - INFO - Chain [1] done processing
16:45:06 - cmdstanpy - INFO - Chain [1] start processing
16:45:06 - cmdstanpy - INFO - Chain [1] done processing
16:45:06 - cmdstanpy - INFO - Chain [1] start processing
16:45:06 - cmdstanpy - INFO - Chain [1] done processing
16:45:06 - cmdstanpy - INFO - Chain [1] start processing
16:45:07 - cmdstanpy - INFO - Chain [1] done processing
16:45:07 - cmdstanpy - INFO - Chain [1] start processing
16:45:07 - cmdstanpy - INFO - Chain [1] done processing
16:45:07 - cmdstanpy - INFO - Chain [1] start processing
16:45:07 - cmdstanpy - INFO - Chain [1] done processing
16:45:07 - cmdstanpy - INFO - Chain [1] 

KeyboardInterrupt: 